# 피처 엔지니어링 EDA

`train_cleaned.csv`와 `test_cleaned.csv`에서 최종 피처 파일을 바로 생성한다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train_cleaned.csv"
TEST_PATH = DATA_DIR / "test_cleaned.csv"
TRAIN_OUT = DATA_DIR / "train_eda_revised.csv"
TEST_OUT = DATA_DIR / "test_eda_revised.csv"

DATE_COLS = ["baseline_create_date", "due_in_date", "clear_date"]
TEXT_DTYPES = {
    "cust_number": "string",
    "business_code": "string",
    "name_customer": "string",
    "cust_payment_terms": "string",
    "cust_payment_terms_grp": "string",
}

LATE_BUSINESS_DAYS = 5
N_RECENT = 5



## 데이터 로드

고객번호처럼 앞자리 0이 중요할 수 있는 컬럼은 문자열로 읽고, 기존 `target`은 비교용으로 `target_old`에 보관한다.


In [ ]:
def load_invoice_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=TEXT_DTYPES)
    if "target" in df.columns:
        df = df.rename(columns={"target": "target_old"})

    # 날짜 컬럼은 이후 영업일 계산에 바로 쓸 수 있게 변환한다.
    for col in DATE_COLS:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


train_eda_revised = load_invoice_data(TRAIN_PATH)
test_eda_revised = load_invoice_data(TEST_PATH)

for name, df in [("train", train_eda_revised), ("test", test_eda_revised)]:
    print(f"{name}: rows={len(df):,}, cols={len(df.columns)}, customers={df['cust_number'].nunique():,}")
    display(df.head(3))



## 기본 피처 생성

만기일과 결제일 사이의 영업일 지연, 새 타깃, 주말 만기 여부, 금액 구간을 만든다.


In [ ]:
def business_days_between(start_dates, end_dates) -> pd.Series:
    start_dates = pd.to_datetime(start_dates, errors="coerce")
    if not isinstance(end_dates, pd.Series):
        end_dates = pd.Series(end_dates, index=start_dates.index)
    end_dates = pd.to_datetime(end_dates, errors="coerce")

    result = pd.Series(np.nan, index=start_dates.index, dtype="float")
    valid = start_dates.notna() & end_dates.notna()
    if valid.any():
        start_np = start_dates.loc[valid].values.astype("datetime64[D]")
        end_np = end_dates.loc[valid].values.astype("datetime64[D]")
        result.loc[valid] = np.busday_count(start_np, end_np)
    return result


def add_business_days_late(df: pd.DataFrame) -> None:
    # 주말을 제외한 실제 지연일을 계산한다.
    df["business_days_late"] = business_days_between(df["due_in_date"], df["clear_date"]).astype(int)


def add_basic_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    for df in [train_df, test_df]:
        add_business_days_late(df)
        df["target"] = (df["business_days_late"] > LATE_BUSINESS_DAYS).astype(int)
        df["due_weekend_flag"] = df["due_in_date"].dt.weekday.isin([5, 6]).astype(int)

    # 금액 구간은 train 기준으로 만들고 test에는 같은 경계를 적용한다.
    _, raw_bins = pd.qcut(train_df["amount_in_usd"], q=4, labels=False, retbins=True, duplicates="drop")
    amount_bins = np.r_[-np.inf, raw_bins[1:-1], np.inf]
    amount_labels = list(range(len(amount_bins) - 1))

    for df in [train_df, test_df]:
        df["amount_bin"] = pd.cut(
            df["amount_in_usd"],
            bins=amount_bins,
            labels=amount_labels,
            include_lowest=True,
        ).astype("int64")

    return amount_bins


amount_bins = add_basic_features(train_eda_revised, test_eda_revised)
print("Amount bin edges:", amount_bins)
print(pd.DataFrame({
    "train_target_old": train_eda_revised["target_old"].value_counts().sort_index(),
    "train_target": train_eda_revised["target"].value_counts().sort_index(),
    "test_target_old": test_eda_revised["target_old"].value_counts().sort_index(),
    "test_target": test_eda_revised["target"].value_counts().sort_index(),
}).fillna(0).astype(int))



## 고객 이력 기반 피처

현재 인보이스 생성 시점에서 알 수 있는 과거 정보만 사용해 고객별 결제 습관을 만든다.


In [ ]:
def add_business_days(start_dates, n_business_days: int) -> pd.Series:
    start_dates = pd.to_datetime(start_dates, errors="coerce")
    result = pd.Series(pd.NaT, index=start_dates.index)
    valid = start_dates.notna()
    if valid.any():
        start_np = start_dates.loc[valid].values.astype("datetime64[D]")
        result.loc[valid] = pd.to_datetime(np.busday_offset(start_np, n_business_days, roll="forward"))
    return result


def known_history_for_date(customer_history: pd.DataFrame, current_date) -> pd.DataFrame:
    prior = customer_history[customer_history["baseline_create_date"] < current_date].copy()
    if prior.empty:
        return prior

    paid_before_current = prior["clear_date"].notna() & (prior["clear_date"] < current_date)
    unpaid_as_of_current = prior["clear_date"].isna() | (prior["clear_date"] >= current_date)
    unpaid_late_days = business_days_between(prior["due_in_date"], current_date)
    known_late_unpaid = unpaid_as_of_current & (unpaid_late_days >= LATE_BUSINESS_DAYS)

    # 이미 결제됐거나 현재 시점에 지연 확정인 과거 인보이스만 사용한다.
    known = prior[paid_before_current | known_late_unpaid].copy()
    if known.empty:
        return known

    paid_late_days = business_days_between(known["due_in_date"], known["clear_date"])
    known_late_unpaid = known_late_unpaid.reindex(known.index).fillna(False)

    known["_known_late_target"] = np.where(
        known_late_unpaid,
        1.0,
        known["target"].astype(float),
    )
    known["_known_late"] = np.where(
        known_late_unpaid,
        1.0,
        (paid_late_days >= LATE_BUSINESS_DAYS).astype(float),
    )
    known["_days_late_as_of_current"] = np.where(
        known_late_unpaid,
        unpaid_late_days.reindex(known.index),
        paid_late_days,
    )

    # 최근 이력 정렬에 사용할 '지연 여부를 알게 된 날짜'를 만든다.
    known["_known_event_date"] = known["clear_date"]
    known.loc[known_late_unpaid, "_known_event_date"] = add_business_days(
        known.loc[known_late_unpaid, "due_in_date"],
        LATE_BUSINESS_DAYS,
    )
    return known


def add_customer_history_features(current_df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    features = [
        "cust_allowed_pay_days_late_rate_past",
        "ratio_paid_invoices_late_past",
        "avg_days_late_paid_late_past",
        "sum_outstanding_amount_past",
        "recent_5_late_rate",
    ]
    current_df[features] = 0.0

    for cust, current_rows_for_customer in current_df.groupby("cust_number", sort=False):
        customer_history = history_df[history_df["cust_number"].eq(cust)]

        for current_date, current_rows in current_rows_for_customer.groupby("baseline_create_date", sort=True):
            known = known_history_for_date(customer_history, current_date)
            if not known.empty:
                rate_by_allowed_days = known.groupby("Allowed_Pay_Days")["_known_late_target"].mean()
                current_df.loc[current_rows.index, "cust_allowed_pay_days_late_rate_past"] = (
                    current_rows["Allowed_Pay_Days"].map(rate_by_allowed_days).fillna(0.0).astype(float)
                )
                current_df.loc[current_rows.index, "ratio_paid_invoices_late_past"] = float(known["_known_late"].mean())

                known_late = known[known["_known_late"].eq(1.0)]
                if not known_late.empty:
                    current_df.loc[current_rows.index, "avg_days_late_paid_late_past"] = float(
                        known_late["_days_late_as_of_current"].mean()
                    )

                recent_known = known.sort_values(
                    ["_known_event_date", "baseline_create_date", "due_in_date", "clear_date"]
                ).tail(N_RECENT)
                current_df.loc[current_rows.index, "recent_5_late_rate"] = float(recent_known["_known_late"].mean())

            # 현재 시점에 아직 결제되지 않은 과거 인보이스 금액 합계.
            outstanding = customer_history[
                (customer_history["baseline_create_date"] < current_date)
                & (customer_history["clear_date"].isna() | (customer_history["clear_date"] >= current_date))
            ]
            current_df.loc[current_rows.index, "sum_outstanding_amount_past"] = max(
                float(outstanding["amount_in_usd"].sum()),
                0.0,
            )

    return current_df


train_eda_revised = add_customer_history_features(train_eda_revised, train_eda_revised)
test_history = pd.concat([train_eda_revised, test_eda_revised], axis=0, ignore_index=True)
test_eda_revised = add_customer_history_features(test_eda_revised, test_history)

history_features = [
    "cust_allowed_pay_days_late_rate_past",
    "ratio_paid_invoices_late_past",
    "avg_days_late_paid_late_past",
    "sum_outstanding_amount_past",
    "recent_5_late_rate",
]
train_eda_revised[history_features].describe()



## 저장 및 검증

중간 산출물 없이 최종 파일만 저장하고, 행 수와 주요 컬럼 상태를 확인한다.


In [ ]:
train_eda_revised.to_csv(TRAIN_OUT, index=False)
test_eda_revised.to_csv(TEST_OUT, index=False)

for name, path, df in [
    ("train", TRAIN_OUT, train_eda_revised),
    ("test", TEST_OUT, test_eda_revised),
]:
    print(f"Saved {name}: {path}")
    print(f"rows={len(df):,}, cols={len(df.columns)}, business_day_gap={'business_day_gap' in df.columns}")

